# Zero-shot predicate retrieval for FDA 510(k) devices

Given a new device's **Indications for Use + Device Description**, rank earlier 510(k) clearances so that the
predicates its summary cites come first. This notebook rebuilds the dataset from public FDA records and runs the
untuned retrieval baselines (lexical, dense embeddings, hybrids). COMS 4705 project: *Retrieving 510(k) Predicate
Devices with Fine-Tuned Text Embeddings* (K. Azami, J. Lamm, S. Chukka).

**Pipeline** (each stage is resumable; re-run a cell after a disconnect and it skips finished work)

1. openFDA 510(k) metadata (176k records) -> summary PDFs from accessdata.fda.gov
2. PyMuPDF text extraction, optional GPU OCR for image-only scans; regex section headers; cited K-numbers with predicate/reference labels
3. optional *closure fetch*: cited predicates older than the corpus window are downloaded too
4. dataset: K-numbers masked, substantial-equivalence sentences stripped; train 2010-2023 / dev 2024 / test 2025+
5. baselines + metrics (Recall@10, Hit@k, MRR with bootstrap CIs); report by review panel and candidate-pool size

**Data source.** `DATA_SOURCE = "drive"` (default) is hermetic with respect to data: it reads the prebuilt v2
bundle (`fdasearch_data_v2` on Google Drive, 670 MB: openFDA export, extracted PDF text incl. OCR and closure docs,
the corpus/query tables and the `predx` export, sha256-verified against `manifest.json`), so nothing is fetched from
FDA and the dataset is byte-identical to the reference run. Models are still downloaded from Hugging Face and
embedded on the Colab GPU (Gemini via the API). `DATA_SOURCE = "fda"` rebuilds the data from openFDA and
accessdata.fda.gov instead.

**Two modes.** `quick` restricts everything to one advisory committee (default Cardiovascular, ~6k documents;
about 10 minutes on a T4 from the bundle, ~15 minutes more when rebuilding). `full` uses all panels (~49k documents;
lexical methods in minutes, each dense model 20-40 minutes on a T4, hours more when rebuilding from FDA).

**Reference results from the full run on an RTX 4090** (same code; numbers in a quick run differ because the
candidate pool is one panel):

Test split (n=4912 subjects cleared 2025-2026), no fine-tuning:

| method | Recall@10 | Hit@1 | Hit@10 | Hit@100 | MRR |
|---|---|---|---|---|---|
| TF-IDF (1-2 grams, cosine) | 69.3 | 49.7 | 82.5 | 95.4 | 0.61 |
| RRF TF-IDF + Gemini | 69.9 | 48.4 | 82.6 | 96.1 | 0.61 |
| RRF TF-IDF + EmbeddingGemma | 69.1 | 48.4 | 81.9 | 96.0 | 0.60 |
| BM25 | 65.4 | 47.3 | 78.4 | 93.3 | 0.58 |
| Gemini embedding-001 | 65.1 | 45.3 | 77.6 | 94.2 | 0.56 |
| EmbeddingGemma-300M | 62.9 | 45.4 | 75.5 | 93.1 | 0.56 |
| bge-m3 | 61.8 | 45.4 | 74.6 | 91.4 | 0.56 |
| e5-base-v2 | 61.8 | 43.3 | 74.3 | 92.1 | 0.54 |
| Qwen3-Embedding-4B | 58.1 | 37.2 | 70.8 | 92.2 | 0.49 |
| LSA (SVD-300) | 56.3 | 38.0 | 68.9 | 89.6 | 0.48 |
| product-code popularity | 19.2 | 5.9 | 25.5 | 69.1 | 0.13 |

Lexical search wins zero-shot; EmbeddingGemma and Gemini are the best untuned embedders; fusion helps Hit@100.
About 9% of the regex "primary predicate" labels disagree with an LLM read of the summary, so all scores are lower bounds.

## 0. Settings

In [ ]:
DATA_SOURCE = "drive"       # "drive" = prebuilt v2 bundle from Google Drive (hermetic); "fda" = rebuild from FDA downloads
DATA_DIR    = "/content/drive/MyDrive/fdasearch_data_v2"   # bundle location after drive.mount (if it was shared with you: Drive > Organize > Add shortcut to My Drive)
DATA_FOLDER_ID = "1Ac0fRIa2dSDRngPj8Pql8eduyMIQY6bk"        # fallback: fetch the folder by id with gdown (needs "anyone with the link" sharing)

MODE       = "quick"        # "quick" = one advisory committee; "full" = all panels
PANEL      = "CV"           # quick mode only. Codes: OR Orthopedic, RA Radiology, CV Cardiovascular, SU Surgery, DE Dental, HO Hospital, GU Gastro/Urology, NE Neurology, ...
MIN_YEAR   = 2010           # corpus window start (summaries before ~2000 are mostly scans)
RUN_CLOSURE = True          # fda source only: also fetch cited predicates outside the window so they become retrievable
RUN_OCR    = False          # fda source only: OCR image-only PDFs (needs a GPU runtime; ~0.7 s/page on T4, ~7 s/page on CPU)
WORKERS    = 2              # extraction processes (free Colab has 2 vCPUs)

# Dense models, downloaded from Hugging Face and run on the Colab GPU. EmbeddingGemma is gated: accept the license at
# https://huggingface.co/google/embeddinggemma-300m and put a token in Colab secrets as HF_TOKEN.
EMBED_MODELS = ["google/embeddinggemma-300m", "BAAI/bge-m3"]      # [] to skip dense models
RUN_GEMINI   = True         # Gemini embedding-001 via the API; needs GEMINI_API_KEY in Colab secrets (skipped if absent)

import os
def secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    return os.environ.get(name, "")
HF_TOKEN = secret("HF_TOKEN"); GEMINI_API_KEY = secret("GEMINI_API_KEY")
print("HF token:", "yes" if HF_TOKEN else "no (EmbeddingGemma will be skipped)", "| Gemini key:", "yes" if GEMINI_API_KEY else "no (skipped)")

## 1. Environment

In [ ]:
%pip install -q "pymupdf>=1.24" "pandas>=2.2" pyarrow "bm25s>=0.2" PyStemmer "scikit-learn>=1.4" "sentence-transformers>=5.0" "transformers>=4.56" google-genai requests
if RUN_OCR:
    %pip install -q rapidocr-onnxruntime onnxruntime-gpu
import os, sys, json, glob, subprocess, pathlib
ROOT = pathlib.Path("/content/fdasearch"); (ROOT / "scripts").mkdir(parents=True, exist_ok=True)
for d in ["data/raw", "data/pdfs", "results"]: (ROOT / d).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
def sh(cmd):
    # run a shell command and stream its output into the cell (Colab does not reliably show inherited stdio)
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True); tail.append(line); tail = tail[-40:]
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}); last lines:\n" + "".join(tail[-15:]))
import torch; print("python", sys.version.split()[0], "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Pipeline scripts
The cells below write the project's scripts into `/content/fdasearch/scripts/`. They are the exact files used for
the reference results (generated from the repo by `scripts/make_colab.py`).

In [ ]:
%%writefile scripts/download_pdfs.py
"""Download 510(k) summary PDFs from accessdata.fda.gov, newest first.

Usage: python download_pdfs.py [--min-year 2019] [--workers 6]
Writes data/pdfs/K######.pdf and data/pdfs/_manifest.tsv (k, status, bytes).
Skips files already present or already recorded as 404.
"""
import argparse, json, os, sys, time, zipfile, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(ROOT, "data", "raw", "device-510k.json.zip")
OUT = os.path.join(ROOT, "data", "pdfs")
MAN = os.path.join(OUT, "_manifest.tsv")
UA = "fdasearch-510k-research/0.1 (academic course project)"


def pdf_url(k):
    yy = int(k[1:3])
    folder = "pdf" if (yy >= 76 or yy == 0) else f"pdf{yy}"
    return f"https://www.accessdata.fda.gov/cdrh_docs/{folder}/{k}.pdf"


OPENFDA_URL = "https://download.open.fda.gov/device/510k/device-510k-0001-of-0001.json.zip"


def load_records():
    if not os.path.exists(RAW):
        os.makedirs(os.path.dirname(RAW), exist_ok=True)
        print("downloading openFDA 510(k) export ...", flush=True)
        r = requests.get(OPENFDA_URL, headers={"User-Agent": UA}, timeout=600)
        r.raise_for_status()
        open(RAW, "wb").write(r.content)
    z = zipfile.ZipFile(RAW)
    return json.load(z.open(z.namelist()[0]))["results"]


def panel_match(r, panel):
    """True if the record belongs to the advisory committee given as code (CV) or description substring."""
    if not panel:
        return True
    p = panel.lower()
    return r.get("advisory_committee", "").lower() == p or p in r.get("advisory_committee_description", "").lower()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--min-year", type=int, default=1976)
    ap.add_argument("--max-year", type=int, default=2100)
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--include-blank", action="store_true", help="also try records with blank statement_or_summary")
    ap.add_argument("--panel", default="", help="only this advisory committee (code like CV, or description substring)")
    a = ap.parse_args()
    os.makedirs(OUT, exist_ok=True)
    recs = load_records()
    done = set()
    if os.path.exists(MAN):
        for line in open(MAN):
            p = line.rstrip("\n").split("\t")
            if len(p) >= 2 and p[1] in ("200", "404"):
                done.add(p[0])
    todo = []
    for r in recs:
        k = r["k_number"]
        if not k.startswith("K"):
            continue
        s = r.get("statement_or_summary", "")
        if s != "Summary" and not (a.include_blank and s == ""):
            continue
        y = int(r["decision_date"][:4])
        if y < a.min_year or y > a.max_year or not panel_match(r, a.panel):
            continue
        if k in done or os.path.exists(os.path.join(OUT, k + ".pdf")):
            continue
        todo.append((r["decision_date"], k))
    todo.sort(reverse=True)
    print(f"{len(todo)} to download ({len(done)} already in manifest)", flush=True)
    lock = threading.Lock()
    man = open(MAN, "a")
    sess = requests.Session()
    sess.headers["User-Agent"] = UA

    def fetch(k):
        url = pdf_url(k)
        for attempt in range(4):
            try:
                r = sess.get(url, timeout=60, allow_redirects=True)
            except Exception as e:
                time.sleep(2 * (attempt + 1)); continue
            if "abuse-detection" in r.url:
                time.sleep(30 * (attempt + 1)); continue
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                tmp = os.path.join(OUT, k + ".part")
                with open(tmp, "wb") as f:
                    f.write(r.content)
                os.replace(tmp, os.path.join(OUT, k + ".pdf"))
                return k, "200", len(r.content)
            if r.status_code == 404:
                return k, "404", 0
            time.sleep(2 * (attempt + 1))
        return k, "ERR", 0

    n_ok = n_404 = n_err = 0
    t0 = time.time()
    with ThreadPoolExecutor(a.workers) as ex:
        futs = [ex.submit(fetch, k) for _, k in todo]
        for i, f in enumerate(as_completed(futs), 1):
            k, st, nb = f.result()
            with lock:
                man.write(f"{k}\t{st}\t{nb}\n"); man.flush()
            if st == "200": n_ok += 1
            elif st == "404": n_404 += 1
            else: n_err += 1
            if i % 200 == 0:
                el = time.time() - t0
                print(f"{i}/{len(todo)} ok={n_ok} 404={n_404} err={n_err} {i/el:.1f}/s", flush=True)
    print(f"done ok={n_ok} 404={n_404} err={n_err} in {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/extract_text.py
"""Extract text, sections, and cited K-numbers from downloaded 510(k) PDFs.

Usage: python extract_text.py [--workers 8]
Reads data/pdfs/K*.pdf, writes data/extracted.jsonl (one record per K; incremental: skips Ks already present).
"""
import argparse, glob, json, os, re, sys
from multiprocessing import Pool

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PDFS = os.path.join(ROOT, "data", "pdfs")
OUTF = os.path.join(ROOT, "data", "extracted.jsonl")
OCR_MIN_CHARS_PER_PAGE = 100   # below this, the PDF is treated as image-only and OCR'd (if --ocr)
OCR_MAX_PAGES = 12
_OCR = None


def ocr_pages(path, dpi=150, max_pages=OCR_MAX_PAGES):
    """OCR an image-only PDF with RapidOCR (ONNX, CUDA if available). Returns list of page texts.
    Lines are ordered top-to-bottom, left-to-right so section headers stay on their own lines."""
    global _OCR
    if _OCR is None:
        import torch  # noqa: F401  loads the CUDA 12 runtime DLLs that onnxruntime-gpu links against
        from rapidocr_onnxruntime import RapidOCR
        _OCR = RapidOCR(det_use_cuda=True, rec_use_cuda=True, cls_use_cuda=True)
    import pymupdf, numpy as np
    out = []
    for page in pymupdf.open(path)[:max_pages]:
        pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csGRAY)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)
        res, _ = _OCR(img)
        if not res:
            out.append(""); continue
        res = sorted(res, key=lambda r: (round(r[0][0][1] / 12), r[0][0][0]))
        out.append("\n".join(r[1] for r in res))
    return out

K_RE = re.compile(r"\b[KkPpDd](?:EN|en)?\s?\d{6}\b")
HDR_WORDS = r"(?:510\s*\(?k\)?\s*summary|submitter|applicant|sponsor|device(?:\s+trade)?\s+name|predicate(?:\s+device)?s?|reference\s+device?s?|device\s+description|description\s+of\s+(?:the\s+)?device|indications?\s+for\s+use|intended\s+use|technological\s+characteristics|comparison|substantial\s+equivalence|performance\s+(?:data|testing)|non-?clinical|clinical|conclusion|summary\s+of|regulatory\s+information|classification|common\s+name)"
HDR_RE = re.compile(r"(?im)^[ \t]*(?:[IVX]+[\.\)]?\s*|\d+(?:\.\d+)?[\.\)]?\s*|[A-Z][\.\)]\s*)?(" + HDR_WORDS + r")[^\n]{0,40}$")


def norm_k(s):
    return s.upper().replace(" ", "")


def section_after(text, pat, maxlen=4000):
    """Return body text following the first header matching pat (regex on header word), up to next header."""
    for m in HDR_RE.finditer(text):
        if re.fullmatch(pat, m.group(1).strip(), flags=re.I):
            start = m.end()
            nxt = HDR_RE.search(text, start + 1)
            end = nxt.start() if nxt else len(text)
            body = text[start:end].strip()
            if len(body) > 40:
                return body[:maxlen]
    return ""


def ifu_from_form(text):
    """Indications for Use from FORM FDA 3881 block."""
    m = re.search(r"Indications for Use\s*\(Describe\)\s*(.*?)(?:Type of Use|Prescription Use|PLEASE DO NOT WRITE|CONTINUE ON A SEPARATE PAGE)", text, flags=re.S | re.I)
    if m:
        body = re.sub(r"\s+", " ", m.group(1)).strip()
        if len(body) > 20:
            return body[:4000]
    return ""


def process(path, ocr=False):
    k = os.path.basename(path)[:-4]
    rec = {"k": k}
    try:
        import pymupdf
        d = pymupdf.open(path)
        pages = [p.get_text() for p in d]
        rec["n_pages"] = len(pages)
        text = "\n".join(pages)
        rec["ocr"] = False
        if ocr and len(text) < OCR_MIN_CHARS_PER_PAGE * max(1, len(pages)):
            pages = ocr_pages(path)
            text = "\n".join(pages)
            rec["ocr"] = True
            rec["ocr_pages"] = len(pages)
    except Exception as e:
        rec["error"] = str(e)[:200]
        return rec
    rec["n_chars"] = len(text)
    # locate 510(k) summary portion (after the decision letter + IFU form)
    ms = list(re.finditer(r"510\s*\(?k\)?\s*summary", text, flags=re.I))
    sum_start = 0
    for m in ms:
        # choose the first occurrence that is at line start-ish (header), skip letter boilerplate
        line_start = text.rfind("\n", 0, m.start()) + 1
        if len(text[line_start:m.start()].strip()) <= 12:
            sum_start = line_start
            break
    summary = text[sum_start:]
    rec["summary_start"] = sum_start
    rec["summary_text"] = re.sub(r"[ \t]+", " ", summary)[:12000]
    rec["ifu_form"] = ifu_from_form(text)
    rec["ifu_sec"] = section_after(summary, r"indications?\s+for\s+use|intended\s+use")
    rec["desc_sec"] = section_after(summary, r"device\s+description|description\s+of\s+(?:the\s+)?device")
    # cited K numbers in order, with predicate/reference labeling by nearest preceding cue word
    cited, labels = [], {}
    for m in K_RE.finditer(summary):
        kk = norm_k(m.group(0))
        if kk == k or kk in labels:
            continue
        window = summary[max(0, m.start() - 700):m.start()].lower()
        ip, ir = window.rfind("predicate"), window.rfind("reference device")
        labels[kk] = "reference" if ir > ip else "predicate"
        cited.append(kk)
    rec["cited"] = cited
    rec["cited_labels"] = labels
    # also any K numbers in the whole doc (letter can mention none; IFU form has own K)
    allk = [norm_k(m.group(0)) for m in K_RE.finditer(text)]
    rec["cited_anywhere"] = [x for x in dict.fromkeys(allk) if x != k]
    return rec


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--pdf-dir", default=PDFS)
    ap.add_argument("--out", default=OUTF)
    ap.add_argument("--ocr", action="store_true", help="OCR image-only PDFs on the GPU (runs sequentially in-process)")
    ap.add_argument("--redo-low-text", action="store_true",
                    help="drop records in --out whose text was below the OCR threshold and extract them again")
    a = ap.parse_args()
    done, low, keep = set(), set(), []
    if os.path.exists(a.out):
        for line in open(a.out, encoding="utf-8"):
            try:
                r = json.loads(line)
            except Exception:
                continue
            done.add(r["k"])
            npages = max(1, r.get("n_pages") or 1)
            if a.redo_low_text and not r.get("ocr") and r.get("n_chars", 0) < OCR_MIN_CHARS_PER_PAGE * npages:
                low.add(r["k"])
            else:
                keep.append(line)
    if low:
        open(a.out, "w", encoding="utf-8").writelines(keep)
        done -= low
        print(f"{len(low)} low-text records removed for re-extraction", flush=True)
    files = [f for f in sorted(glob.glob(os.path.join(a.pdf_dir, "K*.pdf")), reverse=True) if os.path.basename(f)[:-4] not in done]
    if a.limit:
        files = files[: a.limit]
    print(f"{len(files)} new PDFs to extract ({len(done)} done)", flush=True)
    import time
    t0 = time.time()
    with open(a.out, "a", encoding="utf-8") as out:
        if a.ocr or a.workers == 1:
            pool, it = None, (process(f, ocr=a.ocr) for f in files)
        else:
            pool = Pool(a.workers); it = pool.imap_unordered(process, files, chunksize=16)
        for i, rec in enumerate(it, 1):
            out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
            if i % (50 if a.ocr else 1000) == 0:
                print(i, f"{(time.time() - t0) / i:.2f}s/doc", flush=True)
        if pool:
            pool.close(); pool.join()
    print("done", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/build_dataset.py
"""Join openFDA metadata with extracted PDF text; build corpus + query splits.

Outputs data/corpus.parquet (one row per device with usable text) and data/queries.parquet
(subjects from 2024 [dev] and 2025-2026 [test] with >=1 predicate found in the corpus).
"""
import json, os, re, zipfile, collections
import pandas as pd

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(ROOT, "data", "raw", "device-510k.json.zip")
EXT = os.path.join(ROOT, "data", "extracted.jsonl")
EXT_CLOSURE = os.path.join(ROOT, "data", "extracted_closure.jsonl")   # cited predicates outside 2010+ (team's "closure fetch")
TRAIN_FROM = 2010   # train = subjects TRAIN_FROM..2023, dev = 2024, test = 2025+
SE_RE = re.compile(r"(?i)predicate|substantial(?:ly)?[ -]equivalen|510\s*\(?k\)?|reference device")
SENT_RE = re.compile(r"(?<=[.;:])\s+")
K_RE = re.compile(r"\b[KkPpDd](?:EN|en)?\s?\d{6}\b")
WS = re.compile(r"\s+")
MIN_CHARS = 200


def clean(s, maxlen):
    """Mask K-numbers and drop sentences that talk about predicates / substantial equivalence
    (the team's SE_LANGUAGE_IN_INPUT leak check). Returns (text, n_sentences_dropped)."""
    s = K_RE.sub(" ", s or "")
    s = WS.sub(" ", s).strip()
    sents = SENT_RE.split(s)
    keep = [x for x in sents if not SE_RE.search(x)]
    dropped = len(sents) - len(keep)
    if not keep:          # everything mentioned predicates: keep the masked text rather than nothing
        keep, dropped = sents, 0
    return " ".join(keep)[:maxlen], dropped


def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--panel", default="", help="restrict corpus and queries to one advisory committee (code like CV, or description substring)")
    a = ap.parse_args()
    z = zipfile.ZipFile(RAW)
    recs = json.load(z.open(z.namelist()[0]))["results"]
    meta = {}
    for r in recs:
        o = r.get("openfda", {})
        meta[r["k_number"]] = dict(
            k=r["k_number"], date=r["decision_date"], product_code=r.get("product_code", ""),
            device_name=r.get("device_name", ""), applicant=r.get("applicant", ""),
            panel=r.get("advisory_committee_description", ""), specialty=o.get("medical_specialty_description", ""),
            class_name=o.get("device_name", ""), regulation=o.get("regulation_number", ""),
            device_class=o.get("device_class", ""), decision=r.get("decision_code", ""),
            kind=r.get("statement_or_summary", ""), panel_code=r.get("advisory_committee", ""),
        )
    if a.panel:
        p = a.panel.lower()
        meta = {k: m for k, m in meta.items() if m["panel_code"].lower() == p or p in m["panel"].lower()}
        print(f"panel filter {a.panel!r}: {len(meta)} devices", flush=True)
    rows = []
    stats = collections.Counter()
    lines = list(open(EXT, encoding="utf-8"))
    if os.path.exists(EXT_CLOSURE):
        lines += list(open(EXT_CLOSURE, encoding="utf-8"))
    seen = set()
    for line in lines:
        e = json.loads(line)
        k = e["k"]
        if k in seen:
            continue
        seen.add(k)
        if k not in meta or e.get("error"):
            stats["no_meta_or_error"] += 1; continue
        m = dict(meta[k])
        ifu = e.get("ifu_sec") or e.get("ifu_form") or ""
        desc = e.get("desc_sec") or ""
        summ = e.get("summary_text") or ""
        if (e.get("n_chars") or 0) < MIN_CHARS:
            stats["too_little_text"] += 1; continue
        stats["has_ifu" if ifu else "no_ifu"] += 1
        stats["has_desc" if desc else "no_desc"] += 1
        ifu_c, d1 = clean(ifu, 3000); desc_c, d2 = clean(desc, 4000); summ_c, d3 = clean(summ, 6000)
        stats["se_sentences_dropped"] += d1 + d2
        m.update(
            ifu=ifu_c, desc=desc_c, summary=summ_c, se_dropped=d1 + d2,
            n_pages=e.get("n_pages"), n_chars=e.get("n_chars"), ocr=bool(e.get("ocr")),
            cited=e.get("cited", []),
            predicates=[c for c in e.get("cited", []) if e["cited_labels"].get(c) == "predicate"],
            references=[c for c in e.get("cited", []) if e["cited_labels"].get(c) == "reference"],
        )
        # retrieval text: name + IFU + description; fallback to summary body
        core = " ".join(x for x in [m["device_name"], m["ifu"], m["desc"]] if x)
        m["text"] = core if (ifu or desc) else " ".join([m["device_name"], m["summary"][:4000]])
        m["text_source"] = "sections" if (ifu or desc) else "summary"
        rows.append(m)
    corpus = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    corpus.to_parquet(os.path.join(ROOT, "data", "corpus.parquet"))
    print("corpus:", len(corpus), dict(stats))
    print(corpus.groupby(corpus.date.str[:4]).size().to_dict())
    print("text_source:", corpus.text_source.value_counts().to_dict(), "ocr docs:", int(corpus.ocr.sum()))
    name_of = dict(zip(corpus.k, corpus.device_name.str.lower()))

    in_corpus = dict(zip(corpus.k, corpus.date))
    qrows = []
    cov = collections.Counter()
    for _, r in corpus.iterrows():
        y = int(r.date[:4])
        if y < TRAIN_FROM:
            continue
        preds = list(r.predicates)
        if not preds:
            cov["no_predicate_cited"] += 1; continue
        valid = [p for p in preds if p in in_corpus and in_corpus[p] < r.date]
        cov["pred_total"] += len(preds); cov["pred_in_corpus"] += len(valid)
        cov["pred_missing_meta"] += sum(1 for p in preds if p not in in_corpus)
        if not valid:
            cov["no_valid_predicate"] += 1; continue
        qrows.append(dict(
            k=r.k, date=r.date, product_code=r.product_code, specialty=r.specialty, panel=r.panel,
            split="train" if y < 2024 else ("dev" if y == 2024 else "test"), query=r.text, text_source=r.text_source,
            predicates_all=preds, positives=valid, primary=preds[0] if preds[0] in valid else None,
            references=[p for p in r.references if p in in_corpus and in_corpus[p] < r.date],
            # leak flags (team's audit): predicate device name appearing verbatim in the query text
            leak_name=any(len(name_of.get(p, "")) >= 12 and name_of[p] in r.text.lower() for p in valid),
            se_dropped=int(r.se_dropped),
        ))
    q = pd.DataFrame(qrows)
    q.to_parquet(os.path.join(ROOT, "data", "queries.parquet"))
    print("queries:", len(q), q.split.value_counts().to_dict(), dict(cov))
    print("coverage of cited predicates by corpus: %.1f%%" % (100 * cov["pred_in_corpus"] / max(1, cov["pred_total"])))
    print("positives per query:", q.positives.map(len).describe().to_dict())
    for sp in ["train", "dev", "test"]:
        g = q[q.split == sp]
        if len(g):
            print(f"{sp}: n={len(g)} coverage-of-cited={g.positives.map(len).sum() / max(1, g.predicates_all.map(len).sum()):.3f} "
                  f"leak_name={g.leak_name.mean():.3f} se_dropped>0={(g.se_dropped > 0).mean():.3f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/fetch_closure.py
"""Download summary PDFs for cited predicates that are missing from the corpus (the team's 'closure fetch' step).
Writes to data/pdfs_closure/ and data/pdfs_closure/_manifest.tsv. Candidates = K-numbers cited as predicates by any
query that have no usable text in data/corpus.parquet."""
import os, re, sys, time, collections, concurrent.futures as cf
import pandas as pd, requests
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from download_pdfs import pdf_url, UA

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
OUT = os.path.join(ROOT, "data", "pdfs_closure"); os.makedirs(OUT, exist_ok=True)
MAN = os.path.join(OUT, "_manifest.tsv")


def main():
    q = pd.read_parquet(os.path.join(ROOT, "data", "queries.parquet"))
    c = pd.read_parquet(os.path.join(ROOT, "data", "corpus.parquet"))
    have = set(c.k)
    want = sorted({p for ps in q.predicates_all for p in ps if p.startswith("K") and p not in have})
    done = {}
    if os.path.exists(MAN):
        for line in open(MAN):
            k, st = line.rstrip("\n").split("\t")[:2]; done[k] = st
    todo = [k for k in want if k not in done]
    print(f"{len(want)} missing predicates, {len(todo)} to fetch", flush=True)
    sess = requests.Session(); sess.headers["User-Agent"] = UA

    def fetch(k):
        src = os.path.join(ROOT, "data", "pdfs", k + ".pdf")
        dst = os.path.join(OUT, k + ".pdf")
        if os.path.exists(src):
            import shutil; shutil.copy(src, dst); return k, "ok-local"
        for attempt in range(3):
            try:
                r = sess.get(pdf_url(k), timeout=60)
                if r.status_code == 404: return k, "404"
                if r.status_code == 200 and r.content.startswith(b"%PDF"):
                    open(dst, "wb").write(r.content); return k, "ok"
                if b"abuse" in r.content[:4000].lower(): time.sleep(30)
            except Exception as e:
                time.sleep(2)
        return k, "err"

    st = collections.Counter(done.values())
    with open(MAN, "a") as man, cf.ThreadPoolExecutor(1) as ex:
        for i, (k, s) in enumerate(ex.map(fetch, todo), 1):
            man.write(f"{k}\t{s}\n"); man.flush(); st[s] += 1
            if i % 200 == 0: print(i, dict(st), flush=True)
    print("done", dict(st))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/run_baselines.py
"""Zero-shot predicate retrieval baselines.

Usage: python run_baselines.py --methods bm25,lsa,pop,emb:BAAI/bge-base-en-v1.5,ollama:embeddinggemma [--pc]
Writes results/metrics.csv (appends) and results/ranks/{method}.json (rank of each positive per query).
"""
import argparse, json, os, re, sys, time, math
import numpy as np, pandas as pd

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RES = os.path.join(ROOT, "results")
os.makedirs(os.path.join(RES, "ranks"), exist_ok=True)
os.makedirs(os.path.join(RES, "emb"), exist_ok=True)

# model-specific prompts (query, document)
PROMPTS = {
    "intfloat/e5-base-v2": ("query: ", "passage: "),
    "intfloat/e5-large-v2": ("query: ", "passage: "),
    "BAAI/bge-base-en-v1.5": ("Represent this sentence for searching relevant passages: ", ""),
    "BAAI/bge-large-en-v1.5": ("Represent this sentence for searching relevant passages: ", ""),
    "BAAI/bge-m3": ("", ""),
    "nomic-ai/nomic-embed-text-v1.5": ("search_query: ", "search_document: "),
    "Qwen/Qwen3-Embedding-0.6B": ("Instruct: Given a medical device description, retrieve FDA 510(k) predicate devices that are substantially equivalent to it\nQuery: ", ""),
    "Qwen/Qwen3-Embedding-4B": ("Instruct: Given a medical device description, retrieve FDA 510(k) predicate devices that are substantially equivalent to it\nQuery: ", ""),
    "Alibaba-NLP/gte-modernbert-base": ("", ""),
    "sentence-transformers/all-MiniLM-L6-v2": ("", ""),
    "allenai/specter2_base": ("", ""),
    "Snowflake/snowflake-arctic-embed-m-v2.0": ("query: ", ""),
    "google/embeddinggemma-300m": ("task: search result | query: ", "title: none | text: "),
}
OLLAMA_PROMPTS = {"embeddinggemma": ("task: search result | query: ", "title: none | text: "),
                  "nomic-embed-text": ("search_query: ", "search_document: ")}


def load():
    corpus = pd.read_parquet(os.path.join(ROOT, "data", "corpus.parquet"))
    queries = pd.read_parquet(os.path.join(ROOT, "data", "queries.parquet"))
    queries = queries[queries.split.isin(["dev", "test"])].reset_index(drop=True)  # train split is for fine-tuning only
    return corpus, queries


def embed_st(model_name, texts, is_query, batch=64, max_len=512):
    from sentence_transformers import SentenceTransformer
    import torch
    kw = {"trust_remote_code": True}
    if "Qwen3" in model_name or "gemma" in model_name:
        # bf16 where the GPU supports it (Ampere+); older GPUs such as Colab's T4 fall back to fp32
        kw["model_kwargs"] = {"torch_dtype": torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32}
    m = SentenceTransformer(model_name, device="cuda", **kw)
    m.max_seq_length = max_len
    qp, dp = PROMPTS.get(model_name, ("", ""))
    pre = qp if is_query else dp
    out = m.encode([pre + t for t in texts], batch_size=batch, normalize_embeddings=True,
                   show_progress_bar=True, convert_to_numpy=True)
    del m; torch.cuda.empty_cache()
    return out.astype(np.float32)


def _ollama_post(model, chunk, tries=5):
    import requests
    for a in range(tries):
        try:
            r = requests.post("http://localhost:11434/api/embed", json={"model": model, "input": chunk, "truncate": True}, timeout=600)
            if r.status_code == 200:
                return r.json()["embeddings"]
            err = f"{r.status_code} {r.text[:120]}"
        except Exception as e:
            err = str(e)[:120]
        time.sleep(3 * (a + 1))
    if len(chunk) > 1:  # split and retry halves
        h = len(chunk) // 2
        return _ollama_post(model, chunk[:h]) + _ollama_post(model, chunk[h:])
    if len(chunk[0]) > 1500:  # single bad item: shorten
        return _ollama_post(model, [chunk[0][:1500]])
    print("  ollama giving up on one item:", err, flush=True)
    return [None]


def embed_ollama(model, texts, is_query, batch=32):
    qp, dp = OLLAMA_PROMPTS.get(model.split(":")[0], ("", ""))
    pre = qp if is_query else dp
    out = []
    for i in range(0, len(texts), batch):
        chunk = [pre + t[:6000] for t in texts[i:i + batch]]
        out.extend(_ollama_post(model, chunk))
        if (i // batch) % 50 == 0:
            print(f"  ollama {i}/{len(texts)}", flush=True)
    dim = len(next(v for v in out if v is not None))
    v = np.asarray([x if x is not None else [0.0] * dim for x in out], dtype=np.float32)
    return v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-9)


def embed_gemini(model, texts, is_query, batch=100, dim=3072):
    """Gemini embeddings via google-genai. Key from GEMINI_API_KEY or secrets/gemini_key.txt."""
    from google import genai
    from google.genai import types
    envf = os.path.join(ROOT, ".env")
    if os.path.exists(envf):
        for line in open(envf):
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                v = v.strip().strip('"').strip("'")
                if v and not os.environ.get(k.strip()):
                    os.environ[k.strip()] = v
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not key:
        raise SystemExit("No Gemini key: put GEMINI_API_KEY=... in E:\fdasearch\.env")
    client = genai.Client(api_key=key)
    cfg = types.EmbedContentConfig(task_type="RETRIEVAL_QUERY" if is_query else "RETRIEVAL_DOCUMENT", output_dimensionality=dim)
    out = []
    for i in range(0, len(texts), batch):
        chunk = [t[:8000] for t in texts[i:i + batch]]
        for attempt in range(8):
            try:
                r = client.models.embed_content(model=model, contents=chunk, config=cfg)
                out.extend([e.values for e in r.embeddings]); break
            except Exception as e:
                msg = str(e)[:160]
                if attempt == 7:
                    raise
                time.sleep(min(60, 5 * 2 ** attempt))
                print(f"  gemini retry {attempt+1} at {i}: {msg}", flush=True)
        if (i // batch) % 20 == 0:
            print(f"  gemini {i}/{len(texts)}", flush=True)
    v = np.asarray(out, dtype=np.float32)
    return v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), 1e-9)


def cached_embed(tag, fn, texts, keys, is_query, chunk=4000):
    """Embed only texts whose key is not yet cached; cache keyed by K-number (npz), saved every `chunk` texts."""
    p = os.path.join(RES, "emb", f"{tag}.{'q' if is_query else 'd'}.npz")
    have = {}
    if os.path.exists(p):
        z = np.load(p, allow_pickle=False)
        have = dict(zip(z["keys"].tolist(), z["vecs"]))
    todo = [(k, t) for k, t in zip(keys, texts) if k not in have]
    if todo:
        print(f"  embedding {len(todo)} new of {len(texts)} ({len(have)} cached)", flush=True)
        for i in range(0, len(todo), chunk):
            part = todo[i:i + chunk]
            v = fn([t for _, t in part], is_query)
            for (k, _), vec in zip(part, v):
                have[k] = vec
            np.savez(p, keys=np.array(list(have.keys())), vecs=np.stack(list(have.values())))
    return np.stack([have[k] for k in keys]).astype(np.float32)


def evaluate(scores_fn, corpus, queries, method, pc_filter=False):
    """scores_fn(qi) -> np.array of scores over corpus rows (higher better)."""
    kidx = {k: i for i, k in enumerate(corpus.k)}
    dates = corpus.date.values
    pcs = corpus.product_code.values
    rows = []
    rank_dump = {}
    top = np.full((len(queries), 100), -1, dtype=np.int32)
    for qi, q in enumerate(queries.itertuples()):
        s = np.asarray(scores_fn(qi), dtype=np.float64).copy()
        valid = dates < q.date
        if pc_filter:
            valid &= pcs == q.product_code
        s[~valid] = -np.inf
        if q.k in kidx:
            s[kidx[q.k]] = -np.inf
        order = np.argsort(-s, kind="stable")
        top[qi] = order[:100]
        rank_of = np.empty(len(s), dtype=np.int64); rank_of[order] = np.arange(1, len(s) + 1)
        pos = [kidx[p] for p in q.positives]
        pr = sorted(int(rank_of[i]) for i in pos)
        n_valid = int(valid.sum())
        pr = [r if r <= n_valid else 10 ** 9 for r in pr]  # excluded by filter -> not retrievable
        prim = int(rank_of[kidx[q.primary]]) if isinstance(q.primary, str) else None
        if prim is not None and prim > n_valid: prim = 10 ** 9
        rank_dump[q.k] = {"pos": pr, "primary": prim, "n_valid": n_valid}
        rows.append(dict(
            k=q.k, split=q.split, n_pos=len(pr),
            recall10=sum(r <= 10 for r in pr) / len(pr), recall50=sum(r <= 50 for r in pr) / len(pr),
            hit1=float(pr[0] <= 1), hit10=float(pr[0] <= 10), hit100=float(pr[0] <= 100), mrr=1.0 / pr[0],
            prim_hit1=(None if prim is None else float(prim <= 1)), prim_hit10=(None if prim is None else float(prim <= 10)),
            prim_mrr=(None if prim is None else 1.0 / prim),
        ))
    df = pd.DataFrame(rows)
    json.dump(rank_dump, open(os.path.join(RES, "ranks", f"{method}.json"), "w"))
    os.makedirs(os.path.join(RES, "top"), exist_ok=True)
    np.save(os.path.join(RES, "top", f"{method}.npy"), top)
    df.to_csv(os.path.join(RES, "ranks", f"{method}.per_query.csv"), index=False)
    return df


def rrf_scores_fn(names, n_docs, k=60):
    """Reciprocal rank fusion over saved top-100 lists of previously run methods."""
    tops = [np.load(os.path.join(RES, "top", f"{n}.npy")) for n in names]
    def fn(qi):
        s = np.zeros(n_docs)
        for t in tops:
            idx = t[qi]; idx = idx[idx >= 0]
            s[idx] += 1.0 / (k + np.arange(1, len(idx) + 1))
        return s
    return fn


def bootstrap_ci(x, n=1000, seed=0):
    x = np.asarray(x, dtype=float); rng = np.random.default_rng(seed)
    m = np.array([rng.choice(x, len(x)).mean() for _ in range(n)])
    return np.percentile(m, 2.5), np.percentile(m, 97.5)


def summarize(df, method, note=""):
    out = []
    for split, g in df.groupby("split"):
        rec = dict(method=method, split=split, n=len(g), note=note)
        for c in ["recall10", "recall50", "hit1", "hit10", "hit100", "mrr", "prim_hit1", "prim_hit10", "prim_mrr"]:
            v = g[c].dropna()
            lo, hi = bootstrap_ci(v)
            rec[c] = round(v.mean(), 4); rec[c + "_lo"] = round(lo, 4); rec[c + "_hi"] = round(hi, 4)
        out.append(rec)
    return pd.DataFrame(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--methods", required=True)
    ap.add_argument("--pc", action="store_true", help="also run each method restricted to subject's product code")
    ap.add_argument("--max-len", type=int, default=512)
    ap.add_argument("--limit", type=int, default=0)
    a = ap.parse_args()
    corpus, queries = load()
    if a.limit:
        queries = queries.sample(a.limit, random_state=0).reset_index(drop=True)
    print(f"corpus {len(corpus)} docs, {len(queries)} queries", flush=True)
    doc_texts = corpus.text.tolist(); q_texts = queries["query"].tolist()

    for spec in a.methods.split(","):
        t0 = time.time()
        name = spec
        if spec == "bm25":
            import bm25s, Stemmer
            st = Stemmer.Stemmer("english")
            ct = bm25s.tokenize(doc_texts, stopwords="en", stemmer=st)
            idx = bm25s.BM25(); idx.index(ct)
            qt = bm25s.tokenize(q_texts, stopwords="en", stemmer=st, return_ids=False)
            def fn(qi, qt=qt): return idx.get_scores(qt[qi])
        elif spec == "lsa":
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.decomposition import TruncatedSVD
            from sklearn.preprocessing import normalize
            tv = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, stop_words="english")
            D = tv.fit_transform(doc_texts); Q = tv.transform(q_texts)
            svd = TruncatedSVD(300, random_state=0); Dl = normalize(svd.fit_transform(D)); Ql = normalize(svd.transform(Q))
            def fn(qi): return Ql[qi] @ Dl.T
        elif spec == "tfidf":
            from sklearn.feature_extraction.text import TfidfVectorizer
            tv = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, stop_words="english", ngram_range=(1, 2))
            D = tv.fit_transform(doc_texts); Q = tv.transform(q_texts)
            def fn(qi): return (Q[qi] @ D.T).toarray().ravel()
        elif spec == "pop":
            # citation popularity within product code, counts from citing docs cleared before 2024
            cnt = {}
            for r in corpus.itertuples():
                if r.date < "2024-01-01":
                    for p in r.predicates: cnt[p] = cnt.get(p, 0) + 1
            base = np.array([cnt.get(k, 0) for k in corpus.k], dtype=float)
            recency = np.argsort(np.argsort(corpus.date.values)) / len(corpus)  # tie-break by recency
            pcs = corpus.product_code.values
            def fn(qi): return np.where(pcs == queries.product_code.iloc[qi], base + recency, -1.0)
        elif spec == "random":
            rng = np.random.default_rng(0)
            def fn(qi): return rng.random(len(corpus))
        elif spec.startswith(("emb:", "ollama:", "gemini:")):
            kind, model = spec.split(":", 1)
            tag = model.replace("/", "__")
            if kind == "emb":
                f = lambda t, isq: embed_st(model, t, isq, max_len=a.max_len)
            elif kind == "gemini":
                f = lambda t, isq: embed_gemini(model, t, isq)
            else:
                f = lambda t, isq: embed_ollama(model, t, isq)
            D = cached_embed(tag, f, doc_texts, corpus.k.tolist(), False)
            Q = cached_embed(tag, f, q_texts, queries.k.tolist(), True)
            def fn(qi): return Q[qi] @ D.T
            name = spec
        elif spec.startswith("rrf:"):
            # e.g. rrf:bm25|emb_BAAI__bge-base-en-v1.5  (names as saved under results/top)
            fn = rrf_scores_fn(spec[4:].split("|"), len(corpus))
        else:
            print("unknown", spec); continue
        for pc in ([False, True] if a.pc else [False]):
            mname = name + ("+pc" if pc else "")
            df = evaluate(fn, corpus, queries, re.sub(r"[^A-Za-z0-9_.+-]", "_", mname.replace("/", "__")), pc_filter=pc)
            s = summarize(df, mname)
            s["seconds"] = round(time.time() - t0, 1)
            print(s[["method", "split", "n", "recall10", "hit1", "hit10", "mrr", "prim_hit10"]].to_string(index=False), flush=True)
            p = os.path.join(RES, "metrics.csv")
            s.to_csv(p, mode="a", header=not os.path.exists(p), index=False)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/summarize_results.py
"""Turn results/metrics.csv + per-query files into a markdown report (results/REPORT.md)."""
import os, json, glob
import numpy as np, pandas as pd

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RES = os.path.join(ROOT, "results")


def fmt(r, c):
    return f"{100*r[c]:.1f} [{100*r[c+'_lo']:.1f}, {100*r[c+'_hi']:.1f}]"


def main():
    m = pd.read_csv(os.path.join(RES, "metrics.csv"))
    m = m.drop_duplicates(subset=["method", "split"], keep="last")
    corpus = pd.read_parquet(os.path.join(ROOT, "data", "corpus.parquet"))
    queries = pd.read_parquet(os.path.join(ROOT, "data", "queries.parquet"))
    lines = ["# Zero-shot predicate retrieval baselines", ""]
    lines.append(f"Corpus: {len(corpus)} 510(k) summaries ({corpus.date.min()} to {corpus.date.max()}). "
                 f"Queries: {len(queries)} subjects with at least one cited predicate in the corpus "
                 f"(dev = 2024: {(queries.split=='dev').sum()}, test = 2025-2026: {(queries.split=='test').sum()}). "
                 f"Candidates for each query = corpus devices cleared before the subject's decision date. "
                 f"Numbers are percentages with bootstrap 95% CIs over queries.")
    lines.append("")
    for split in ["test", "dev"]:
        g = m[m.split == split].sort_values("recall10", ascending=False)
        lines += [f"## {split} split (n={int(g.n.iloc[0])})", "",
                  "| method | Recall@10 | Hit@1 | Hit@10 | Hit@100 | MRR | primary Hit@1 | primary Hit@10 |", "|---|---|---|---|---|---|---|---|"]
        for _, r in g.iterrows():
            lines.append(f"| {r.method} | {fmt(r,'recall10')} | {fmt(r,'hit1')} | {fmt(r,'hit10')} | {fmt(r,'hit100')} | {fmt(r,'mrr')} | {fmt(r,'prim_hit1')} | {fmt(r,'prim_hit10')} |")
        lines.append("")
    # per-specialty breakdown for each method on test
    KEEP = ["tfidf", "bm25", "lsa", "pop", "ollama_embeddinggemma", "emb_BAAI__bge-m3", "emb_intfloat__e5-base-v2",
            "emb_Qwen__Qwen3-Embedding-4B", "gemini_gemini-embedding-001", "rrf_tfidf_ollama_embeddinggemma", "rrf_tfidf_gemini_gemini-embedding-001"]
    pq = {}
    for f in glob.glob(os.path.join(RES, "ranks", "*.per_query.csv")):
        name = os.path.basename(f).replace(".per_query.csv", "")
        if name in KEEP:
            pq[name] = pd.read_csv(f)
    if pq:
        qmeta = queries.set_index("k")
        lines += ["## Recall@10 by review panel (test split)", ""]
        tab = {}
        for name, d in pq.items():
            d = d[d.split == "test"].copy()
            d["panel"] = d.k.map(qmeta.panel)
            tab[name] = d.groupby("panel").recall10.mean()
        tab = pd.DataFrame(tab)
        cnt = queries[queries.split == "test"].groupby("panel").size()
        tab.insert(0, "n", cnt)
        tab = tab.sort_values("n", ascending=False)
        lines.append("| panel | n | " + " | ".join(tab.columns[1:]) + " |")
        lines.append("|---|---|" + "---|" * (len(tab.columns) - 1))
        for p, r in tab.iterrows():
            lines.append(f"| {p} | {int(r.n)} | " + " | ".join(f"{100*r[c]:.0f}" for c in tab.columns[1:]) + " |")
        lines.append("")
        # by candidate-pool size within product code
        lines += ["## Recall@10 by number of earlier same-product-code devices in corpus (test split)", ""]
        pc_count = {}
        corp = corpus[["k", "date", "product_code"]].sort_values("date")
        for q in queries.itertuples():
            pc_count[q.k] = int(((corp.product_code == q.product_code) & (corp.date < q.date)).sum())
        bins = [0, 10, 50, 200, 10 ** 9]; labels = ["<10", "10-49", "50-199", "200+"]
        tab = {}
        for name, d in pq.items():
            d = d[d.split == "test"].copy()
            d["bin"] = pd.cut(d.k.map(pc_count), bins=bins, labels=labels, right=False)
            tab[name] = d.groupby("bin", observed=True).recall10.mean()
        tab = pd.DataFrame(tab)
        qt = queries[queries.split == "test"]
        tab.insert(0, "n", pd.cut(qt.k.map(pc_count), bins=bins, labels=labels, right=False).value_counts())
        lines.append("| same-code candidates | n | " + " | ".join(tab.columns[1:]) + " |")
        lines.append("|---|---|" + "---|" * (len(tab.columns) - 1))
        for p, r in tab.iterrows():
            lines.append(f"| {p} | {int(r.n)} | " + " | ".join(f"{100*r[c]:.0f}" for c in tab.columns[1:]) + " |")
        lines.append("")
    open(os.path.join(RES, "REPORT.md"), "w", encoding="utf-8").write("\n".join(lines))
    print("\n".join(lines))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/export_predx.py
"""Export data/corpus.parquet + data/queries.parquet in the team's predx dataset layout so their training code
can read it: data/predx/{corpus.jsonl.gz, queries_{train,dev,test}.jsonl.gz, train_pairs.jsonl.gz, dataset_card.json}.
Field names follow the records printed by predx_colab_test_2.ipynb (cell 25)."""
import os, json, gzip, collections, datetime
import pandas as pd

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
OUT = os.path.join(ROOT, "data", "predx"); os.makedirs(OUT, exist_ok=True)


def jl(path, rows):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def main():
    c = pd.read_parquet(os.path.join(ROOT, "data", "corpus.parquet"))
    q = pd.read_parquet(os.path.join(ROOT, "data", "queries.parquet"))
    date = dict(zip(c.k, c.date)); code = dict(zip(c.k, c.product_code))
    corpus_rows = [dict(k=r.k, date=r.date, year=int(r.date[:4]), product_code=r.product_code, panel=r.panel,
                        device_name=r.device_name, applicant=r.applicant, ifu=r.ifu, description=r.desc,
                        n_pages=int(r.n_pages) if pd.notna(r.n_pages) else None, ocr=bool(r.ocr),
                        text_source=r.text_source) for r in c.itertuples()]
    jl(os.path.join(OUT, "corpus.jsonl.gz"), corpus_rows)
    counts = {}
    pairs = []
    for split in ["train", "dev", "test"]:
        g = q[q.split == split]
        rows = []
        for r in g.itertuples():
            cr = c.loc[c.k == r.k].iloc[0]
            gold = []
            for i, p in enumerate(r.predicates_all):
                inc = p in date and date[p] < r.date
                gold.append(dict(k=p, role="primary" if i == 0 else "additional", in_corpus=bool(inc),
                                 reason=None if inc else ("no_text" if p not in date else "not_earlier"),
                                 year=(int(date[p][:4]) if p in date else None), product_code=code.get(p)))
            n_cand = int((c.date < r.date).sum()); n_same = int(((c.date < r.date) & (c.product_code == r.product_code)).sum())
            rows.append(dict(k=r.k, split=split, date=r.date, year=int(r.date[:4]), product_code=r.product_code, panel=r.panel,
                             device_name=cr.device_name, ifu=cr.ifu, description=cr.desc,
                             gold=gold, reference=list(r.references), gold_in_corpus=list(r.positives),
                             gold_primary_in_corpus=[r.primary] if isinstance(r.primary, str) else [],
                             n_candidates=n_cand, n_same_code_candidates=n_same, retrievable=len(r.positives) > 0,
                             leak=dict(residual_k=0, pred_language=False, gold_name=bool(r.leak_name), se_sentences_dropped=int(r.se_dropped))))
            if split == "train":
                for p in r.positives:
                    pairs.append(dict(query_k=r.k, doc_k=p, role="primary" if p == r.primary else "additional"))
        jl(os.path.join(OUT, f"queries_{split}.jsonl.gz"), rows)
        counts[split] = len(rows)
    jl(os.path.join(OUT, "train_pairs.jsonl.gz"), pairs)
    card = dict(built=datetime.date.today().isoformat(), n_corpus=len(c), counts=counts, n_train_pairs=len(pairs),
                mask_level="k_numbers+se_sentences", corpus_years=[int(c.date.min()[:4]), int(c.date.max()[:4])],
                ocr_docs=int(c.ocr.sum()), source="fdasearch pipeline (PyMuPDF + RapidOCR), openFDA export 2026-11-18")
    json.dump(card, open(os.path.join(OUT, "dataset_card.json"), "w"), indent=1)
    print(card)


if __name__ == "__main__":
    main()

## 3. Data
**drive**: the bundle is copied from Drive (mounted, or fetched by folder id), every file is checked against the
sha256 in `manifest.json` and the extracted text is unpacked. In `quick` mode the dataset is rebuilt for one panel
from the extracted text (no PDFs needed); in `full` mode the shipped corpus/query tables are used as they are.

**fda**: the downloader sends an identifying research User-Agent (FDA's server rejects anonymous clients) and runs
at ~25 PDFs/s with 6 workers. ~0.6% of summaries have no PDF (404).

In [ ]:
import hashlib, shutil, gzip, zipfile
panel_arg = f"--panel {PANEL}" if MODE == "quick" else ""

def fetch_bundle():
    src = pathlib.Path(DATA_DIR)
    if not src.exists():
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception as e:
            print("Drive mount not available:", e)
    if not src.exists() and DATA_FOLDER_ID:
        sh("pip install -q gdown")
        src = pathlib.Path("/content/fdasearch_data_v2")
        sh(f"gdown --folder https://drive.google.com/drive/folders/{DATA_FOLDER_ID} -O {src}")
    assert (src / "manifest.json").exists(), f"bundle not found at {src}: mount Drive with fdasearch_data_v2 in My Drive, or set DATA_FOLDER_ID"
    man = json.load(open(src / "manifest.json"))
    place = {"device-510k.json.zip": "data/raw/device-510k.json.zip", "extracted.jsonl.gz": "data/extracted.jsonl.gz",
             "extracted_closure.jsonl.gz": "data/extracted_closure.jsonl.gz", "corpus.parquet": "data/corpus_v2.parquet",
             "queries.parquet": "data/queries_v2.parquet", "predx.zip": "data/predx_v2.zip"}
    for f in man["files"]:
        rel = f["path"]
        dst = ROOT / (place.get(rel) or ("results/emb/" + rel[4:] if rel.startswith("emb/") else "results/reference/" + rel.split("/")[-1] if rel.startswith("reference_results/") else rel))
        if dst.exists() and dst.stat().st_size == f["bytes"]:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        print(f"  {rel} ({f['bytes']/1e6:.0f} MB)", flush=True)
        shutil.copyfile(src / rel, dst)
        h = hashlib.sha256()
        with open(dst, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
        assert h.hexdigest() == f["sha256"], f"checksum mismatch for {rel}"
    for gz in ["data/extracted.jsonl.gz", "data/extracted_closure.jsonl.gz"]:
        out = ROOT / gz[:-3]
        if not out.exists():
            with gzip.open(ROOT / gz, "rb") as fi, open(out, "wb") as fo: shutil.copyfileobj(fi, fo, 1 << 24)
    if MODE == "quick":
        sh(f"python scripts/build_dataset.py {panel_arg}")
    else:
        shutil.copyfile(ROOT / "data/corpus_v2.parquet", ROOT / "data/corpus.parquet")
        shutil.copyfile(ROOT / "data/queries_v2.parquet", ROOT / "data/queries.parquet")
    print("bundle ready")

if DATA_SOURCE == "drive":
    fetch_bundle()
else:
    sh(f"python scripts/download_pdfs.py --min-year {MIN_YEAR} --workers 6 {panel_arg}")
    sh(f"python scripts/extract_text.py --workers {WORKERS}")
    if RUN_OCR:
        sh("python scripts/extract_text.py --ocr --workers 1 --redo-low-text")
    sh(f"python scripts/build_dataset.py {panel_arg}")

### 3b. Closure fetch (fda source only)
The bundle already contains the closure documents. When rebuilding from FDA, predicates cited by the queries but outside the window (mostly 2000s clearances) are downloaded and extracted,
then the dataset is rebuilt so they count as retrievable positives. Older PDFs are often scans: without OCR,
about half of them yield no text and stay unretrievable.

In [ ]:
if RUN_CLOSURE and DATA_SOURCE == "fda":
    sh("python scripts/fetch_closure.py")
    ocr = "--ocr --workers 1" if RUN_OCR else f"--workers {WORKERS}"
    sh(f"python scripts/extract_text.py {ocr} --pdf-dir data/pdfs_closure --out data/extracted_closure.jsonl")
    sh(f"python scripts/build_dataset.py {panel_arg}")
import pandas as pd
q = pd.read_parquet("data/queries.parquet"); c = pd.read_parquet("data/corpus.parquet")
print(f"corpus {len(c)} docs {c.date.min()}..{c.date.max()} | queries", q.split.value_counts().to_dict())

## 4. Baselines
Candidates for each query are the corpus documents cleared before the subject. `--pc` additionally scores each
method restricted to the subject's product code. Embeddings are cached per K-number in `results/emb/`.

In [ ]:
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN            # read by huggingface_hub in the subprocesses below
    from huggingface_hub import login; login(token=HF_TOKEN)
import transformers; print("transformers", transformers.__version__, "(EmbeddingGemma needs >= 4.56)")
if GEMINI_API_KEY:
    open(".env", "w").write(f"GEMINI_API_KEY={GEMINI_API_KEY}\n")

sh("python scripts/run_baselines.py --pc --methods random,pop,bm25,tfidf,lsa")
dense_specs = ["emb:" + m for m in EMBED_MODELS if HF_TOKEN or "gemma" not in m]
if RUN_GEMINI and GEMINI_API_KEY:
    dense_specs.append("gemini:gemini-embedding-001")
if dense_specs:
    sh("python scripts/run_baselines.py --pc --methods " + ",".join(dense_specs))

### 4b. Hybrids
Reciprocal-rank fusion of the saved top-100 lists (`rrf:a|b`; names are the sanitized method names in `results/top/`).

In [ ]:
import re
tags = [re.sub(r"[^A-Za-z0-9_.+-]", "_", m.replace("/", "__")) for m in dense_specs]      # e.g. emb_google__embeddinggemma-300m, gemini_gemini-embedding-001
fusions = [f"rrf:tfidf|{t}" for t in tags] + [f"rrf:bm25|{t}" for t in tags[:1]]
if fusions:
    sh('python scripts/run_baselines.py --methods "' + ",".join(fusions) + '"')

## 5. Report

In [ ]:
sh("python scripts/summarize_results.py > /dev/null")
from IPython.display import Markdown, display
display(Markdown(open("results/REPORT.md").read()))

### Reproducibility check
With `DATA_SOURCE="drive"` and `MODE="full"` the lexical methods must match the reference run to the fourth decimal
(same texts, same code); dense models re-embedded here should land within a few tenths of a point (GPU/batch
nondeterminism). The reference EmbeddingGemma run went through Ollama (`ollama:embeddinggemma`), so it has no
direct counterpart. In `quick` mode the pool is one panel, so numbers differ by design.

In [ ]:
import pandas as pd
ref = "results/reference/metrics.csv"
if os.path.exists(ref):
    a = pd.read_csv("results/metrics.csv").drop_duplicates(["method", "split"], keep="last").set_index(["method", "split"])
    b = pd.read_csv(ref).set_index(["method", "split"])
    common = a.index.intersection(b.index)
    cmp = pd.DataFrame({"this_run": a.loc[common, "recall10"], "reference": b.loc[common, "recall10"]})
    cmp["diff"] = (cmp.this_run - cmp.reference).round(4)
    print(f"{len(common)} (method, split) pairs in common; max |diff| in Recall@10 = {cmp['diff'].abs().max():.4f}")
    display(cmp.sort_values("reference", ascending=False).head(20))

### Look at one query
What the model sees (K-numbers masked, substantial-equivalence sentences removed), the cited predicates, and the
TF-IDF top 10.

In [ ]:
import json, numpy as np
q = pd.read_parquet("data/queries.parquet"); c = pd.read_parquet("data/corpus.parquet")
qt = q[q.split == "test"].reset_index(drop=True)
top = np.load("results/top/tfidf.npy")                      # rows follow the dev+test query order used by run_baselines
qd = q[q.split.isin(["dev", "test"])].reset_index(drop=True)
i = int(qd.index[qd.split == "test"][0])                    # first test query; change i to browse
r = qd.iloc[i]
print(r.k, r.date, r.product_code, "|", r.panel); print(r.query[:600], "...\n")
print("cited predicates:", list(r.predicates_all), "| primary:", r.primary)
for rank, j in enumerate(top[i][:10], 1):
    d = c.iloc[j]; mark = "<-- predicate" if d.k in set(r.positives) else ""
    print(f"{rank:2d}. {d.k} {d.date} {d.product_code} {d.device_name[:60]} {mark}")

## 6. Keep the results
Zips the metrics, report, per-query ranks and the dataset in the team's `predx` layout (`data/predx/`).
Mount Drive first if you want them to survive the runtime.

In [ ]:
sh("python scripts/export_predx.py")
import shutil, zipfile
with zipfile.ZipFile("/content/fdasearch_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in ["results/metrics.csv", "results/REPORT.md", "data/dataset_card.json"]:
        if os.path.exists(p): z.write(p)
    for d in ["results/ranks", "data/predx"]:
        for f in glob.glob(d + "/*"): z.write(f)
print("/content/fdasearch_results.zip", f"{os.path.getsize('/content/fdasearch_results.zip')/1e6:.1f} MB")
try:
    from google.colab import files; files.download("/content/fdasearch_results.zip")
except Exception as e:
    print("not in Colab:", e)
# from google.colab import drive; drive.mount('/content/drive'); shutil.copy('/content/fdasearch_results.zip', '/content/drive/MyDrive/')